In [ ]:
import sys
import os
import warnings
os.environ['OPENBLAS_NUM_THREADS'] = '1'
warnings.filterwarnings('ignore')

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import time
import pyarrow.parquet as pq
import scipy
import implicit
import bisect
import sklearn.metrics as m
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

In [ ]:
LOCAL_DATA_PATH = './data/'
SPLIT_SEED = 42
DATA_FILE = 'competition_data_final_pqt'
TARGET_FILE = 'public_train.pqt'
SUBMISSION_FILE = 'submit_2.pqt'

In [ ]:
id_to_submit = pq.read_table(f'{LOCAL_DATA_PATH}/{SUBMISSION_FILE}').to_pandas()

In [ ]:
data = pq.read_table(f'{LOCAL_DATA_PATH}/{DATA_FILE}')
pd.DataFrame([(z.name, z.type) for z in data.schema], columns = [['field', 'type']])

??????   
, ?????????? ?????   
, ???????????? ??????????   
, ?????? ?????????   
, ?????, ? ???????? ?????? ????????? ??????   
, ??? ?????????? (???????? ??? ???-?? ??????)   
, ??????????? ?? ??????????   
, ?????? ???? ??????????   
, ????   
, ????? ??? (????, ????? ...)   
, ????? ????????   
, id ????????????   

In [ ]:
data.select(['cpe_type_cd']).to_pandas()['cpe_type_cd'].value_counts()

In [ ]:
targets = pq.read_table(f'{LOCAL_DATA_PATH}/{TARGET_FILE}')
pd.DataFrame([(z.name, z.type) for z in targets.schema], columns = [['field', 'type']])

In [ ]:
%%time
df_urls = pq.read_table(f'{LOCAL_DATA_PATH}/{DATA_FILE}').select(['user_id', 'url_host']).to_pandas()
df_urls = df_urls.groupby('url_host').agg({'user_id': lambda x: x.nunique()}).reset_index()

In [ ]:
urls_filter = set(df_urls[df_urls.user_id>1]['url_host'])
len(urls_filter)

In [ ]:
%%time
data_agg = data.select(['user_id', 'url_host', 'request_cnt']).\
    group_by(['user_id', 'url_host']).aggregate([('request_cnt', "sum")])

In [ ]:
%%time
data_agg = data_agg.to_pandas()
data_agg = data_agg[data_agg['url_host'].isin(urls_filter)]

In [ ]:
url_set = set(data_agg['url_host'])
print(f'{len(url_set)} urls')
url_dict = {url: idurl for url, idurl in zip(url_set, range(len(url_set)))}
usr_set = set(data_agg['user_id'])
print(f'{len(usr_set)} users')
usr_dict = {usr: user_id for usr, user_id in zip(usr_set, range(len(usr_set)))}

In [ ]:
%%time
values = np.array(data_agg['request_cnt_sum'])
rows = np.array(data_agg['user_id'].map(usr_dict))
cols = np.array(data_agg['url_host'].map(url_dict))
mat = scipy.sparse.coo_matrix((values, (rows, cols)), shape=(rows.max() + 1, cols.max() + 1))
als = implicit.approximate_als.FaissAlternatingLeastSquares(factors = 128, iterations = 50, use_gpu = False, \
       calculate_training_loss = False, regularization = 0.1)

In [ ]:
%%time
als.fit(mat)
u_factors = als.model.user_factors 
d_factors = als.model.item_factors

In [ ]:
%%time
inv_usr_map = {v: k for k, v in usr_dict.items()}
usr_emb = pd.DataFrame(u_factors, columns=[f"usr_f_{i}" for i in range(128)])
usr_emb['user_id'] = usr_emb.index.map(inv_usr_map)

In [ ]:
usr_emb.to_csv('./data/user_factors_128.csv', index=False)

In [ ]:
# %%time
inv_url_map = {v: k for k, v in url_dict.items()}
url_emb = pd.DataFrame(d_factors, columns=[f"url_f_{i}" for i in range(128)])
url_emb['url_host'] = url_emb.index.map(inv_url_map)

In [ ]:
# %%time
df_usr_url = data.select(['user_id', 'url_host']).to_pandas()
df_usr_url.head()

In [ ]:
# %%time

# import numpy as np
# df_url_embeds = df_usr_url.merge(url_emb, on="url_host")
# del df_url_embeds['url_host']
# df_url_embeds = df_url_embeds.groupby(['user_id'])[[f"url_f_{i}" for i in range(50)]].agg([np.mean, np.max]).reset_index()
# df_url_embeds.columns = [f"{u}_{v}" if u!='user_id' else u for u,v in df_url_embeds.columns.to_flat_index()]
# # df_url_embeds.to_csv('./data/url_factors.csv', index=False)
# df_url_embeds.head(5)

In [ ]:
# df_url_embeds.to_csv('./data/url_factors.csv', index=False)

## ??????? ?????? ?? ????

In [ ]:
%%time
inv_usr_map = {v: k for k, v in usr_dict.items()}
usr_emb = pd.DataFrame(u_factors)
usr_emb['user_id'] = usr_emb.index.map(inv_usr_map)
usr_targets = targets.to_pandas()
df = usr_targets.merge(usr_emb, how = 'inner', on = ['user_id'])
# df = df.merge(df_url_embeds, how = 'inner', on = ['user_id'])
df = df[df['is_male'] != 'NA']
df = df.dropna()
df['is_male'] = df['is_male'].map(int)
df['is_male'].value_counts()

In [ ]:
%%time
x_train, x_test, y_train, y_test = train_test_split(\
    df.drop(['user_id', 'age', 'is_male'], axis = 1), df['is_male'], test_size = 0.33, random_state = SPLIT_SEED)
clf = CatBoostClassifier()
clf.fit(x_train, y_train, verbose = False)
print(f'GINI ?? ???? {2 * m.roc_auc_score(y_test, clf.predict_proba(x_test)[:,1]) - 1:2.3f}')

In [ ]:
#clf.fit(df.drop(['user_id', 'age', 'is_male'], axis = 1), df['is_male'], verbose = False)
#id_to_submit['is_male'] = clf.predict_proba(id_to_submit.merge(usr_emb, how = 'inner', on = ['user_id']))[:,1]

## ??????? ?????? ?? ????????

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

#%matplotlib inline
#sns.set_style('darkgrid')

In [ ]:
def age_bucket(x):
    return bisect.bisect_left([18,25,35,45,55,65], x)

In [ ]:
df = usr_targets.merge(usr_emb, how = 'inner', on = ['user_id'])
df = df[df['age'] != 'NA']
df = df.dropna()
df['age'] = df['age'].map(age_bucket)
sns.histplot(df['age'], bins = 7)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(\
    df.drop(['user_id', 'age', 'is_male'], axis = 1), df['age'], test_size = 0.33, random_state = SPLIT_SEED)

clf = CatBoostClassifier()
clf.fit(x_train, y_train, verbose = False)
print(m.classification_report(y_test, clf.predict(x_test), \
                            target_names = ['<18', '18-25','25-34', '35-44', '45-54', '55-65', '65+']))

In [ ]:
#clf.fit(df.drop(['user_id', 'age', 'is_male'], axis = 1), df['age'], verbose = False)
#id_to_submit['age'] = clf.predict(id_to_submit[['user_id']].merge(usr_emb, how = 'inner', on = ['user_id']))

## ??????

In [ ]:
#id_to_submit.head()

In [ ]:
# id_to_submit.to_csv(f'{LOCAL_DATA_PATH}/submission.csv', index = False)

In [ ]:
#! head $LOCAL_DATA_PATH/submission.csv

# ???? ?? ??????????

In [ ]:
# context_scorer(submission, answers)